# CrOSSD

## Jupyterlite stuff

In [ ]:
from pyodide.http import pyxhr
cors = "https://corsproxy.io/?url="

## Website Example

In [ ]:
from IPython.display import IFrame, display

display(IFrame("https://health.crossd.tech", 1050,700))

## API Endpoints

### `/api/projects`
Get a list of all projects

In [ ]:
#import requests
import json
from rich import print

project = "lorabridge/lorabridge"

# response = requests.post('https://health.crossd.tech/api/projects')
# Synchronous HTTP request
response = pyxhr.post(cors + "https://health.crossd.tech/api/projects")
data = response.json()

In [ ]:
print(f"Project count: {len(data)}\n")

In [ ]:
print(json.dumps(data,indent=4))

### `/api/snapshots`
Get a list of snapshots (timestamps of the particular retrievals) of the specified project

In [ ]:
headers = {
    'Content-Type': 'application/json',
}

json_data = {
    'term': project,
}

#response = requests.post('https://health.crossd.tech/api/snapshots', headers=headers, json=json_data)
response = pyxhr.post(cors + 'https://health.crossd.tech/api/snapshots', headers=headers, json=json_data)
data = response.json()

In [ ]:
print(f"Snapshot for project {json_data['term']}")
print(f"Snapshot count: {len(data)}")

In [ ]:
print(json.dumps(data,indent=4))

### `/api/metrics`
Get the calculated metrics for the specified project at the desired point in time (snapshot)

In [ ]:
json_data = {
    'term': project,
    'timestamp': data[-1],
}

# response = requests.post('https://health.crossd.tech/api/metrics', headers=headers, json=json_data)
response = pyxhr.post(cors + 'https://health.crossd.tech/api/metrics', headers=headers, json=json_data)
data = response.json()

In [ ]:
print(json.dumps(data, indent = 4))

### `/api/repo`
Get the retrieved raw data of the repository at the specified point in time (snapshot)

In [ ]:
json_data = {
    'term': project,
    'timestamp': json_data['timestamp'],
}

# response = requests.post('https://health.crossd.tech/api/repo', headers=headers, json=json_data)
response = pyxhr.post(cors + 'https://health.crossd.tech/api/repo', headers=headers, json=json_data)
data = response.json()

In [ ]:
print(json.dumps(data, indent = 4))

## Pipenv dependency example

### Pipfile Content

In [21]:
from rich.markup import escape
print(escape(open("Pipfile").read()))

[[source]]
url = "https://pypi.org/simple"
verify_ssl = true
name = "pypi"

[packages]
notebook = "*"
rich = "*"

[dev-packages]

[requires]
python_version = "3.13"



### Retrieve Github URIs from Pipfile dependencies

In [ ]:
import tomllib
import re

# read dependencies
packages = tomllib.loads(open("Pipfile").read())["packages"].keys()
projects=[]

for pkg in packages:
    # get pypi info for each package
    #response = requests.get(f"https://pypi.org/pypi/{pkg}/json")
    response = pyxhr.get(f"https://pypi.org/pypi/{pkg}/json")
    data = response.json()
    # Check if there are any github URIs
    for name, url in data["info"]["project_urls"].items():
        if m:=re.match(r"https://github\.com/([a-zA-Z0-9-_]+)/([a-zA-Z0-9-_]+)",url):
            projects.append("/".join(m.group(1,2)))
            break
print(projects)


### Retrieve snapshots for the projects in Pipfile

In [ ]:
snapshots={}

for project in projects:
    json_data = {
    'term': project,
    }
    
    #response = requests.post('https://health.crossd.tech/api/snapshots', headers=headers, json=json_data)
    response = pyxhr.post(cors + 'https://health.crossd.tech/api/snapshots', headers=headers, json=json_data)
    data = response.json()
    snapshots[project] = data

print(snapshots)

### Retrieve metrics for latest snapshot of the projects

In [ ]:
metrics = {}

for project in snapshots:
    if snapshots[project]:
        json_data = {
            'term': project,
            'timestamp': snapshots[project][-1],
        }

        #response = requests.post('https://health.crossd.tech/api/metrics', headers=headers, json=json_data)
        response = pyxhr.post(cors + 'https://health.crossd.tech/api/metrics', headers=headers, json=json_data)
        data = response.json()
        metrics[project] = data

In [ ]:
print(metrics)